### Importing data

In [ ]:
def add_request_column(df):
    """
    Add a 'request' column to the DataFrame with unique identifiers for each row.

    Parameters:
        df (pd.DataFrame): The input DataFrame to be augmented.

    Returns:
        pd.DataFrame: The augmented DataFrame with a 'request' column.
    """
    df = df.copy()
    df['request'] = ['request_' + str(i + 1) for i in range(len(df))]
    return df


In [ ]:
import pandas as pd

input_df = pd.read_csv('/Users/gabo/Documents/Thesis/OpenAI/Testing/testing_data.csv')


### Creating dictionary for api requests

In [ ]:
def create_api_requests_all_prompt(prompt, df, column_name,crypto,model="gpt-4o-mini", max_tokens=10000):
    """
    Creates a list of API request dictionaries for the OpenAI Batch API.
    
    Parameters:
        prompts (str): The initial prompt describing the task for the model.
        df (pd.DataFrame): A dataframe containing the data for additional requests.
        column_name (str): The column in the dataframe containing the content for user messages.
        model (str, optional): The OpenAI model to be used. Defaults to "gpt-3.5-turbo-0125".
        max_tokens (int, optional): The maximum number of tokens for the response. Defaults to 1000.
        
    Returns:
        list: A list of dictionaries representing API requests.
    """
    api_requests = []
    
    # Add entries for each row in the dataframe
    for idx, row in df.iterrows():        
        api_requests.append({
            "custom_id": row['request'],
            "model": model,
            "messages": [
                {"role": "user", "content": f'{prompt} Comment: {row[column_name]}. Crypto: {crypto}'}
            ],
            "max_tokens": max_tokens
        })
    
    return api_requests

### Creating jsonl file from the dictionary

In [ ]:
import json

def create_jsonl_for_openai_batch(api_requests, output_file):
    """
    Creates a JSONL file for the OpenAI Batch API.
    
    Parameters:
        api_requests (list): A list of dictionaries, where each dictionary contains:
            - custom_id (str): A unique identifier for the request.
            - model (str): The model to be used (e.g., "gpt-3.5-turbo").
            - messages (list): A list of messages in the format [{"role": "system", "content": ...}, {"role": "user", "content": ...}].
            - max_tokens (int, optional): The maximum number of tokens for the response.
        output_file (str): The path to the JSONL output file.
    """
    with open(output_file, 'w') as f:
        for request in api_requests:
            # Build the JSON object for each request
            jsonl_entry = {
                "custom_id": request["custom_id"],
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": request["model"],
                    "messages": request["messages"],
                    "max_tokens": request.get("max_tokens", 1000)  # Default to 1000 if not provided
                }
            }
            # Write the JSON object as a line in the file
            f.write(json.dumps(jsonl_entry) + '\n')


In [ ]:
prompt = 'You are an expert in sentiment analysis of crypto-related comments on Reddit. Analyze the following comment and classify it as bullish, bearish, or neutral based on the sentiment towards Bitcoin. Take into account the unique terminology, slang, and context commonly used in the crypto and Reddit communities. Consider both explicit statements and implied sentiment within the comment. Provide only the sentiment label (bullish/bearish/neutral) as output and nothing else.'
crypto = 'Bitcoin (BTC)'
model = 'gpt-4o-mini'
column_name = 'comment'
max_tokens = 10

api_requests = create_api_requests_all_prompt(prompt, input_df, column_name,crypto,model, max_tokens)

output_file = '/Users/gabo/Documents/Thesis/OpenAI/Comments/Input/testing_data.jsonl'
create_jsonl_for_openai_batch(api_requests, output_file)
print(f"JSONL file created at {output_file}")

In [ ]:
def view_jsonl_file(file_path):
    """
    Reads and displays the content of a .jsonl file.

    Parameters:
        file_path (str): Path to the .jsonl file.

    Returns:
        list: A list of dictionaries representing the parsed JSONL content.
    """
    content = []
    with open(file_path, 'r') as f:
        for line in f:
            # Parse each line as a JSON object
            json_object = json.loads(line.strip())
            content.append(json_object)
    
    return content

# Example usage
jsonl_content = view_jsonl_file(output_file)

# Display the content
for entry in jsonl_content:
    print(json.dumps(entry, indent=4))  # Pretty print each JSON object


### Uploading batch jsonl file to OpenAI

In [ ]:
from openai import OpenAI

client = OpenAI(api_key = 'API_KEY')

batch_file = '/Users/gabo/Documents/Thesis/OpenAI/Comments/Input/testing_data.jsonl'

batch_input_file = client.files.create(
file=open(batch_file, "rb"),
purpose="batch"
)

#file_id = 'file-EmkGCZL6375RxVKU2nU5Ue'

In [ ]:
from openai import OpenAI

client = OpenAI(api_key = 'API_KEY')

name_list =['Bitcoin_Bitcoin_input21_1','Bitcoin_Bitcoin_input21_2']

for name in name_list:
    
    batch_file = f'/Users/gabo/Documents/Thesis/OpenAI/Comments/Input/{name}.jsonl'

    batch_input_file = client.files.create(
    file=open(batch_file, "rb"),
    purpose="batch"
    )

### Creating batch

In [ ]:
batch_input_file_id = batch_input_file.id

client.batches.create(
  input_file_id=batch_input_file_id,
  endpoint="/v1/chat/completions",
  completion_window="24h",
  metadata={
    "description": "testing_data"
  }
)

### Checking status of batch

In [ ]:
client.batches.retrieve("batch_67643b1338a88190bb3d2c36a0b80dd0")

### Downloading batch results

In [ ]:
import json

def extract_predictions_to_df(jsonl_path):
    """
    Extract sentiment predictions from a JSONL file and return a DataFrame.

    Parameters:
        jsonl_path (str): Path to the .jsonl file.

    Returns:
        pd.DataFrame: A DataFrame with 'request' and 'prediction' columns.
    """
    data = []

    with open(jsonl_path, 'r') as file:
        for line in file:
            entry = json.loads(line.strip())  # Load each line as JSON
            custom_id = entry.get("custom_id", "unknown_id")
            
            # Navigate to the sentiment prediction
            try:
                sentiment = entry["response"]["body"]["choices"][0]["message"]["content"].strip().lower()
            except (KeyError, IndexError):
                sentiment = "error_extraction"  # Handle missing or malformed data
            
            data.append({"request": custom_id, "prediction": sentiment})

    # Create a DataFrame
    df = pd.DataFrame(data)
    return df

In [ ]:
batch_output_id = f'testing_data_output'
output_json_file = f'/Users/gabo/Documents/Thesis/OpenAI/Testing/{batch_output_id}.jsonl'
prediction_df = extract_predictions_to_df(output_json_file)
merged_df = filter_df.merge(prediction_df, on='request', how='left')
merged_df.to_csv(f'/Users/gabo/Documents/Thesis/OpenAI/Testing/{batch_output_id}.csv',index=False)
merged_df